# 07 - Track A: CESNET-TLS22 service-category classifier

**Purpose:** Train a gradient-boosted classifier on flow metadata to predict the dataset's own service category, split by day; chart it against baselines.

**Inputs:** CESNET-TLS22 S HDF5 (bounded deterministic sample read in chunks with h5py)

**Outputs:** metric comparison chart, per-class recall chart, confusion-matrix heatmap

**Status labels:** VERIFIED run. **This is a traffic-classification and scale demo. These labels are not cryptographic or PQC labels, and this model says nothing about quantum readiness.**

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
from cryptoshift.datasets import resolve_dataset_paths
from cryptoshift.ml import cesnet_service
h5 = resolve_dataset_paths().cesnet_h5_path
res = cesnet_service.run(h5, chunks_per_day=15, chunk_rows=1000)
res.pop('_model')
print(res['disclaimer'])
pd.Series({k: res[k] for k in ('sample','ppi_rows_identified','model','seed','library_versions')}).to_frame('run')

In [ ]:
m = pd.DataFrame({k: {x: v[x] for x in ('accuracy','macro_f1')} for k, v in res['metrics'].items()}).T.rename(index={'model':'HistGradientBoosting','majority_baseline':'majority class','rule_baseline_bytes_rev_bins':'rule: BYTES_REV bins'})
ax = m.plot(kind='bar', figsize=(9, 4), color=['#457b9d','#e9c46a'], title='Test days 11-17 Oct 2021 (trained on 4-10 Oct)')
for cont in ax.containers: ax.bar_label(cont, fmt='%.3f', fontsize=8)
ax.set_ylim(0, 1); ax.tick_params(axis='x', rotation=0); savefig('07_metrics_vs_baselines')
m

In [ ]:
rec = pd.Series(res['metrics']['model']['per_class_recall']).sort_values()
ax = rec.plot(kind='barh', figsize=(9, 6), color=['#e76f51' if v < 0.6 else '#2a9d8f' for v in rec], title='Per-class recall on the test days (model)')
ax.set_xlim(0, 1); savefig('07_per_class_recall')

In [ ]:
import numpy as np
cmx = np.array(res['metrics']['model']['confusion_matrix_row_normalised']); labs = res['metrics']['model']['confusion_labels']
fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(cmx, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks(range(len(labs)), labs, rotation=90, fontsize=7); ax.set_yticks(range(len(labs)), labs, fontsize=7); ax.grid(False)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('Confusion matrix (row-normalised)'); fig.colorbar(im, ax=ax, fraction=0.04)
savefig('07_confusion_matrix')